In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("=" * 80)
print("[STEP 1] Job parameters received")
print("=" * 80)
print(f"  landing_timestamp : '{landing_timestamp}'")
print(f"  table_name        : '{table_name}'")
print(f"  merge_keys        : '{merge_keys}'")
print(f"  load_type         : '{load_type}'")

In [0]:
source_table = f"adb_caresync.bronze.{table_name}"
filter_condition = f"landing_timestamp ='{landing_timestamp}'"

print("=" * 80)
print("[STEP 2] Reading data from Bronze")
print("=" * 80)
print(f"  Source table     : {source_table}")
print(f"  Filter condition : {filter_condition}")
print("  Query being prepared:")
print(f"    SELECT * FROM {source_table} WHERE {filter_condition}")

bronze_df = spark.read.table(source_table).filter(filter_condition)

bronze_count = bronze_df.count()
print(f"  Records read from bronze : {bronze_count}")
print(f"  Columns ({len(bronze_df.columns)}) : {bronze_df.columns}")
if bronze_count == 0:
    print("  WARNING: No records found for the given landing_timestamp.")

In [0]:
from pyspark.sql.functions import lit, current_timestamp
print("=" * 80)
print("[STEP 3] Adding audit columns")
print("=" * 80)

updated_df = bronze_df.withColumn("insert_timestamp", current_timestamp()).withColumn("update_timestamp", current_timestamp())

print("  Added columns : insert_timestamp = current_timestamp(), update_timestamp = current_timestamp()")
print("  Equivalent query:")
print(f"    SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp")
print(f"    FROM {source_table} WHERE {filter_condition}")
print(f"  Final columns ({len(updated_df.columns)}) : {updated_df.columns}")

In [0]:
from delta.tables import DeltaTable

target_table = f"adb_caresync.silver.{table_name}"
load_type_norm = load_type.strip().upper()

print("=" * 80)
print("[STEP 4] Loading data into Silver")
print("=" * 80)
print(f"  Target table         : {target_table}")
print(f"  Load type (normalized): {load_type_norm}")
print(f"  Target table exists  : {spark.catalog.tableExists(target_table)}")

if load_type_norm == "FULL":
    # Full load: overwrite target table
    print("  Mode: FULL -> overwriting the target table")
    print("  Query being prepared:")
    print(f"    CREATE OR REPLACE TABLE {target_table} AS")
    print(f"    SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp")
    print(f"    FROM {source_table} WHERE {filter_condition}")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print(f"  FULL load completed. Rows now in target: {spark.table(target_table).count()}")

elif load_type_norm == "APPEND":
    # Append: add incoming records to target table
    print("  Mode: APPEND -> adding incoming records to the target table")
    print("  Query being prepared:")
    print(f"    INSERT INTO {target_table}")
    print(f"    SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp")
    print(f"    FROM {source_table} WHERE {filter_condition}")
    updated_df.write.mode("append").saveAsTable(target_table)
    print(f"  APPEND completed. Rows now in target: {spark.table(target_table).count()}")

elif load_type_norm == "MERGE":
    # Merge (upsert): update existing records and insert new ones based on merge keys
    print("  Mode: MERGE -> upserting records based on merge keys")
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    print(f"  Parsed merge keys : {keys}")
    if not keys:
        raise ValueError("merge_keys must be provided for MERGE load type (comma-separated column names)")
    missing = [k for k in keys if k not in updated_df.columns]
    if missing:
        raise ValueError(f"Merge keys not found in source data: {missing}")
    print("  All merge keys found in source data.")

    if not spark.catalog.tableExists(target_table):
        # First run: create the target table
        print("  Target table does not exist -> first run, creating it from source data")
        print("  Query being prepared:")
        print(f"    CREATE TABLE {target_table} AS")
        print(f"    SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp")
        print(f"    FROM {source_table} WHERE {filter_condition}")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print(f"  Target table created. Rows now in target: {spark.table(target_table).count()}")
    else:
        # Deduplicate source on merge keys to avoid multiple-match errors
        source_df = updated_df.dropDuplicates(keys)
        merge_condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in keys])
        # On update, keep the original insert_timestamp
        update_cols = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != "insert_timestamp"}

        source_count = updated_df.count()
        dedup_count = source_df.count()
        print(f"  Source rows before dedup : {source_count}")
        print(f"  Source rows after dedup  : {dedup_count} (removed {source_count - dedup_count} duplicates on {keys})")
        print(f"  Merge condition          : {merge_condition}")
        print(f"  Columns updated on match : {list(update_cols.keys())} (insert_timestamp preserved)")

        update_set_sql = ",\n      ".join([f"t.{k} = {v}" for k, v in update_cols.items()])
        merge_sql = (
            f"    MERGE INTO {target_table} AS t\n"
            f"    USING (\n"
            f"      SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp\n"
            f"      FROM {source_table} WHERE {filter_condition}\n"
            f"      -- deduplicated on {keys}\n"
            f"    ) AS s\n"
            f"    ON {merge_condition}\n"
            f"    WHEN MATCHED THEN UPDATE SET\n"
            f"      {update_set_sql}\n"
            f"    WHEN NOT MATCHED THEN INSERT *"
        )
        print("  Query being prepared:")
        print(merge_sql)

        (DeltaTable.forName(spark, target_table).alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_cols)
            .whenNotMatchedInsertAll()
            .execute())

        # Print MERGE result metrics from the latest Delta history entry
        metrics = DeltaTable.forName(spark, target_table).history(1).select("operationMetrics").collect()[0][0] or {}
        print("  MERGE completed. Operation metrics:")
        print(f"    Rows inserted : {metrics.get('numTargetRowsInserted', 'n/a')}")
        print(f"    Rows updated  : {metrics.get('numTargetRowsUpdated', 'n/a')}")
        print(f"    Rows deleted  : {metrics.get('numTargetRowsDeleted', 'n/a')}")
        print(f"    Source rows   : {metrics.get('numSourceRows', 'n/a')}")
        print(f"  Rows now in target: {spark.table(target_table).count()}")

else:
    print(f"  ERROR: Unsupported load_type '{load_type}'")
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected FULL, APPEND or MERGE.")

In [0]:
record_count = updated_df.count()
print("=" * 80)
print("[STEP 5] Notebook completed")
print("=" * 80)
print(f"  Table           : {table_name}")
print(f"  Load type       : {load_type}")
print(f"  Records processed: {record_count}")
print(f"  Returning record count '{record_count}' to the caller via dbutils.notebook.exit")
dbutils.notebook.exit(str(record_count))